# EDA: Credit Card Fraud Data

This notebook performs exploratory data analysis on the bank credit card transaction fraud dataset.

## Objectives
1. Data exploration and understanding
2. Feature analysis (V1-V28 PCA features)
3. Class imbalance analysis
4. Statistical summaries


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')

# Set up paths - ensure we're working from the project root
project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
elif (project_root / 'notebooks').exists():
    pass  # Already at project root
else:
    # Try to find project root by looking for data directory
    current = Path().resolve()
    while current != current.parent:
        if (current / 'data').exists():
            project_root = current
            break
        current = current.parent

DATA_DIR = project_root / 'data' / 'raw'
OUTPUT_DIR = project_root / 'outputs' / 'eda' / 'creditcard'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Create output directory if it doesn't exist

# Print relative paths so output is the same for all team members
print(f"Project root: .")
print(f"Data directory: {DATA_DIR.relative_to(project_root)}")
print(f"Output directory: {OUTPUT_DIR.relative_to(project_root)}")
print(f"Data directory exists: {DATA_DIR.exists()}")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Helper function to save plots (prints relative path for portability)
def save_plot(fig, filename, dpi=300, bbox_inches='tight'):
    """Save plot to output directory"""
    filepath = OUTPUT_DIR / filename
    fig.savefig(filepath, dpi=dpi, bbox_inches=bbox_inches)
    rel = filepath.relative_to(project_root)
    print(f"Plot saved to: {rel}")

print("Libraries imported successfully!")


## 1. Load Data


In [ ]:
# Load the credit card data
cc_df = pd.read_csv(DATA_DIR / 'creditcard.csv')

print(f"Dataset Shape: {cc_df.shape}")
print(f"\nColumns: {cc_df.columns.tolist()}")
print("\nFirst few rows:")
cc_df.head()


## 2. Data Cleaning


In [ ]:
# Check for missing values
print("Missing Values:")
print(cc_df.isnull().sum())
print(f"\nTotal missing values: {cc_df.isnull().sum().sum()}")

# Check for duplicates
print(f"\nDuplicate rows: {cc_df.duplicated().sum()}")

# Check data types
print("\nData Types:")
print(cc_df.dtypes)

# Basic info
print("\nDataset Info:")
cc_df.info()


In [ ]:
# Handle missing values if any
if cc_df.isnull().sum().sum() > 0:
    missing_cols = cc_df.columns[cc_df.isnull().any()].tolist()
    print(f"Columns with missing values: {missing_cols}")
    for col in missing_cols:
        missing_pct = (cc_df[col].isnull().sum() / len(cc_df)) * 100
        print(f"{col}: {cc_df[col].isnull().sum()} ({missing_pct:.2f}%)")
        # Document decision on handling strategy
else:
    print("No missing values found!")

# Remove duplicates if any
initial_shape = cc_df.shape
cc_df = cc_df.drop_duplicates()
final_shape = cc_df.shape

if initial_shape[0] != final_shape[0]:
    print(f"\nRemoved {initial_shape[0] - final_shape[0]} duplicate rows")
else:
    print("\nNo duplicates found")
    
print(f"Final dataset shape: {cc_df.shape}")


## 3. Class Distribution Analysis


In [ ]:
# Analyze class distribution
class_counts = cc_df['Class'].value_counts()
class_proportions = cc_df['Class'].value_counts(normalize=True)

print("Class Distribution:")
print(f"Non-fraudulent (0): {class_counts[0]:,} ({class_proportions[0]*100:.4f}%)")
print(f"Fraudulent (1): {class_counts[1]:,} ({class_proportions[1]*100:.4f}%)")
print(f"\nImbalance Ratio: {class_counts[0]/class_counts[1]:.2f}:1")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
axes[0].bar(['Non-Fraudulent (0)', 'Fraudulent (1)'], class_counts.values, color=['skyblue', 'coral'])
axes[0].set_title('Class Distribution (Counts)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
axes[1].pie(class_counts.values, labels=['Non-Fraudulent (0)', 'Fraudulent (1)'], 
            autopct='%1.4f%%', startangle=90, colors=['skyblue', 'coral'])
axes[1].set_title('Class Distribution (Proportions)')

plt.tight_layout()
save_plot(fig, 'creditcard_class_distribution.png')
plt.show()


## 4. Feature Analysis


In [ ]:
# Summary statistics for all features
print("Summary Statistics:")
print(cc_df.describe())

# Focus on Time and Amount (non-PCA features)
print("\n" + "="*50)
print("Time and Amount Statistics:")
print(cc_df[['Time', 'Amount']].describe())


In [ ]:
# Distribution of Time and Amount
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time distribution
axes[0].hist(cc_df['Time'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Time')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Frequency')

# Amount distribution (log scale might be helpful)
axes[1].hist(cc_df['Amount'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Distribution of Amount')
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Log scale for Amount (if needed)
fig, ax = plt.subplots(figsize=(8, 5))
cc_df['Amount'].hist(bins=50, edgecolor='black', alpha=0.7, ax=ax)
ax.set_yscale('log')
ax.set_title('Distribution of Amount (Log Scale)')
ax.set_xlabel('Amount ($)')
ax.set_ylabel('Frequency (Log Scale)')
plt.show()


In [ ]:
# Analyze PCA features (V1-V28)
pca_features = [f'V{i}' for i in range(1, 29)]

print(f"Number of PCA features: {len(pca_features)}")
print("\nPCA Features Statistics:")
print(cc_df[pca_features].describe().T.head(10))


In [ ]:
# Check for outliers in PCA features
print("Outliers Analysis (using IQR method):")
for col in pca_features[:5]:  # Check first 5 PCA features
    Q1 = cc_df[col].quantile(0.25)
    Q3 = cc_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = cc_df[(cc_df[col] < lower_bound) | (cc_df[col] > upper_bound)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(cc_df)*100:.2f}%)")


## 5. Bivariate Analysis


In [ ]:
# Amount by fraud class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
cc_df.boxplot(column='Amount', by='Class', ax=axes[0])
axes[0].set_title('Amount Distribution by Fraud Class')
axes[0].set_xlabel('Fraud Class')
axes[0].set_ylabel('Amount ($)')

# Violin plot
sns.violinplot(data=cc_df, x='Class', y='Amount', ax=axes[1])
axes[1].set_title('Amount Distribution by Fraud Class (Violin Plot)')
axes[1].set_xlabel('Fraud Class')
axes[1].set_ylabel('Amount ($)')
axes[1].set_yscale('log')  # Log scale for better visualization

plt.tight_layout()
save_plot(fig, 'creditcard_amount_by_class.png')
plt.show()

# Statistical summary by class
print("\nAmount Statistics by Class:")
print(cc_df.groupby('Class')['Amount'].describe())


In [ ]:
# Time by fraud class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cc_df.boxplot(column='Time', by='Class', ax=axes[0])
axes[0].set_title('Time Distribution by Fraud Class')
axes[0].set_xlabel('Fraud Class')
axes[0].set_ylabel('Time (seconds)')

sns.violinplot(data=cc_df, x='Class', y='Time', ax=axes[1])
axes[1].set_title('Time Distribution by Fraud Class (Violin Plot)')
axes[1].set_xlabel('Fraud Class')
axes[1].set_ylabel('Time (seconds)')

plt.tight_layout()
save_plot(fig, 'creditcard_time_by_class.png')
plt.show()

print("\nTime Statistics by Class:")
print(cc_df.groupby('Class')['Time'].describe())


In [ ]:
# Analyze PCA features by class (sample a few features)
sample_pca_features = pca_features[:6]  # Analyze first 6 PCA features

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(sample_pca_features):
    sns.boxplot(data=cc_df, x='Class', y=feature, ax=axes[idx])
    axes[idx].set_title(f'{feature} Distribution by Fraud Class')
    axes[idx].set_xlabel('Fraud Class')
    axes[idx].set_ylabel(feature)

plt.tight_layout()
save_plot(fig, 'creditcard_pca_features_by_class.png')
plt.show()


## 6. Correlation Analysis


In [ ]:
# Correlation matrix (sample for visualization - full matrix might be too large)
# Focus on correlation with target variable
correlations = cc_df.corr()['Class'].sort_values(ascending=False)

print("Features Correlated with Class (Target):")
print(correlations)

# Visualize top correlations
top_corr = correlations.head(10).tail(9)  # Exclude Class itself
bottom_corr = correlations.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(len(top_corr)), top_corr.values, color='coral')
axes[0].set_yticks(range(len(top_corr)))
axes[0].set_yticklabels(top_corr.index)
axes[0].set_xlabel('Correlation with Class')
axes[0].set_title('Top Positive Correlations with Fraud')
axes[0].invert_yaxis()

axes[1].barh(range(len(bottom_corr)), bottom_corr.values, color='steelblue')
axes[1].set_yticks(range(len(bottom_corr)))
axes[1].set_yticklabels(bottom_corr.index)
axes[1].set_xlabel('Correlation with Class')
axes[1].set_title('Top Negative Correlations with Fraud')
axes[1].invert_yaxis()

plt.tight_layout()
save_plot(fig, 'creditcard_correlation_analysis.png')
plt.show()


## 7. Summary and Next Steps

### Key Findings:
1. **Class Imbalance**: [Document the extreme imbalance ratio found]
2. **Missing Values**: [Document any missing values and handling strategy]
3. **Key Patterns**: [Document any notable patterns in PCA features, Amount, Time]
4. **Feature Correlations**: [Document important correlations with target]

### Next Steps:
- Feature engineering (if needed)
- Data transformation (scaling)
- Handle class imbalance for modeling
